In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
DATA_PATH = "../data/raw/retail_clean_dataset/"
ANOMALY_PATH = "../data/raw/retail_contaminated_dataset/"

customers = pd.read_csv(DATA_PATH + "customer_master.csv")

inventory = pd.read_csv(DATA_PATH + "inventory_snapshot.csv")

promotions = pd.read_csv(DATA_PATH + "promotions.csv")

sales = pd.read_csv(DATA_PATH + "sales_transactions.csv")

products = pd.read_csv(DATA_PATH + "sku_master.csv")

stores = pd.read_csv(DATA_PATH + "store_master.csv")

flags = pd.read_csv(
    ANOMALY_PATH + "sku_inventory_flags.csv"
)

In [7]:
datasets = {
    "customers": customers,
    "inventory": inventory,
    "promotions": promotions,
    "sales": sales,
    "products": products,
    "stores": stores,
    "flags": flags
}

for name, df in datasets.items():
    print("-" * 50)
    print(name.upper())
    print("Shape:", df.shape)

--------------------------------------------------
CUSTOMERS
Shape: (10000, 7)
--------------------------------------------------
INVENTORY
Shape: (21228, 6)
--------------------------------------------------
PROMOTIONS
Shape: (100, 8)
--------------------------------------------------
SALES
Shape: (9972038, 11)
--------------------------------------------------
PRODUCTS
Shape: (5000, 7)
--------------------------------------------------
STORES
Shape: (30, 5)
--------------------------------------------------
FLAGS
Shape: (600, 6)


In [9]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name.upper())

    info = pd.DataFrame({

        "Column": df.columns,

        "Data Type": df.dtypes.values,

        "Missing Values": df.isnull().sum().values,

        "Unique Values": df.nunique().values

    })

    display(info)


CUSTOMERS


,Column,Data Type,Missing Values,Unique Values
0,cust_id,object,0,10000
1,age,int64,0,63
2,gender,object,0,3
3,city,object,0,15
4,loyalty_segment,object,0,4
5,preferred_channel,object,0,3
6,registration_date,object,0,3661



INVENTORY


,Column,Data Type,Missing Values,Unique Values
0,store_id,object,0,30
1,sku_id,object,0,4481
2,stock_on_hand,int64,0,586
3,reorder_point,int64,0,81
4,safety_stock,int64,0,47
5,last_restock_date,object,0,221



PROMOTIONS


,Column,Data Type,Missing Values,Unique Values
0,promo_id,object,0,100
1,promo_name,object,0,72
2,start_date,object,0,96
3,end_date,object,0,97
4,discount_pct,float64,0,87
5,promo_type,object,0,5
6,target_type,object,0,4
7,target_value,object,0,49


... (4 outputs left out)


STORES


,Column,Data Type,Missing Values,Unique Values
0,store_id,object,0,30
1,store_name,object,0,30
2,city,object,0,12
3,store_type,object,0,4
4,opening_date,object,0,30



FLAGS


,Column,Data Type,Missing Values,Unique Values
0,sku_id,object,0,600
1,flag,object,0,2
2,affected_stores,object,0,600
3,window_start,object,400,60
4,window_end,object,400,62
5,notes,object,0,2


In [11]:
for name, df in datasets.items():

    memory = df.memory_usage(deep=True).sum() / 1024**2

    print(f"{name}: {memory:.2f} MB")

customers: 3.77 MB
inventory: 4.39 MB
promotions: 0.05 MB
sales: 4409.08 MB
products: 1.78 MB
stores: 0.01 MB
flags: 0.29 MB


In [13]:
for name, df in datasets.items():

    print("\n", name.upper())

    print(df.isnull().sum())


 CUSTOMERS
cust_id              0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64

 INVENTORY
store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
dtype: int64

 PROMOTIONS
promo_id        0
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
target_type     0
target_value    0
dtype: int64

 SALES
date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7881687
dtype: int64

 PRODUCTS
sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64

 STORES
store_id        0
store_name      0
city    

In [15]:
for name, df in datasets.items():

    print(

        name,

        "duplicates:",

        df.duplicated().sum()

    )

customers duplicates: 0
inventory duplicates: 0
promotions duplicates: 0
sales duplicates: 13019
products duplicates: 0
stores duplicates: 0
flags duplicates: 0


In [17]:
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"]
)

inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"]
)

promotions["start_date"] = pd.to_datetime(
    promotions["start_date"]
)

promotions["end_date"] = pd.to_datetime(
    promotions["end_date"]
)

sales["date"] = pd.to_datetime(
    sales["date"]
)

stores["opening_date"] = pd.to_datetime(
    stores["opening_date"]
)

## Observations

1. Sales table is the fact table.

2. Sales has 10 million rows.

3. Products has 5000 SKUs.

4. There are 30 stores.

5. Promotions contain 100 campaigns.

6. Inventory anomalies exist in contaminated data.

7. sku_inventory_flags is the ground truth.

In [20]:
promo_percentage = (
    sales["promo_id"].notna().mean()
) * 100

print(f"{promo_percentage:.2f}%")

20.96%


In [28]:
duplicate_rows = sales[sales.duplicated(keep=False)]

duplicate_rows.sort_values(
    by=["receipt_id"]
).head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2945,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2946,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5864,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5865,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [24]:
sales[sales.duplicated()].shape

(13019, 11)

In [26]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9972038 entries, 0 to 9972037
Data columns (total 11 columns):
 #   Column        Dtype         
---  ------        -----         
 0   date          datetime64[ns]
 1   receipt_id    object        
 2   store_id      object        
 3   sku_id        object        
 4   customer_id   object        
 5   quantity      int64         
 6   unit_price    float64       
 7   total_value   float64       
 8   channel       object        
 9   discount_pct  float64       
 10  promo_id      object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(6)
memory usage: 836.9+ MB
